In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import warnings
warnings.filterwarnings("ignore")
file_list = [
    "1_global_power_plant_database.csv",
    "2_global_ccs_database.csv",
    "3_smr_database.csv",
    "4_hydrogen.csv",
    "5_net_electricity_imports_database.csv",
    "6_data_center.csv",
    "7_storage.csv",
    "8_gas.csv"
]

In [4]:
df_list = []
name_list = []
for f_name in file_list:
    print(f_name)
    df_list.append(pd.read_csv(f_name,encoding='utf-8'))
    print(df_list[-1].columns)
    
df_list[0] = df_list[0].loc[ df_list[0]['fuel'].isin(['Biomass',"Hydro"])]
df_list[1] = df_list[1].loc[ df_list[1]['Status Name'].isin(["Completed","Active"])]
df_list[3] = df_list[3].loc[ df_list[3]['Status'].isin(['Operational'])]
df_list[4] = df_list[4].loc[ df_list[4]['Net imports - TWh'].astype(float)!=0]

1_global_power_plant_database.csv
Index(['country', 'country_long', 'fuel', 'latitude', 'longitude',
       'capacity_mw', 'name'],
      dtype='object')
2_global_ccs_database.csv
Index(['Project Name', 'Type Name', 'Status Detail Name', 'Status Name',
       'Project Date', 'Company', 'Country Name', 'Location', 'Other', 'Plant',
       'Project Link', 'Unnamed: 11', 'Technology Name', 'Unit Base Name 1',
       'Latitude', 'Longitude', 'Major Injection Amount'],
      dtype='object')
3_smr_database.csv
Index(['name', 'type', 'country', 'status', 'latitude', 'longitude',
       'capacity_mw', 'year', 'link'],
      dtype='object')
4_hydrogen.csv
Index(['Ref', 'Project name', 'Country', 'Status', 'Technology',
       'Technology_electricity', 'Technology_electricity_details', 'Product',
       'EndUse_Refining', 'EndUse_Ammonia', 'EndUse_Methanol',
       'EndUse_Iron&Steel', 'EndUse_Other Ind', 'EndUse_Mobility',
       'EndUse_Power', 'EndUse_Grid inj.', 'EndUse_CHP',
       'EndUse_

In [5]:
concap = pd.read_csv('concap.csv')
# print(concap.head())
c_map = """
Democratic Republic of Congo,Democratic Republic of the Congo
North Macedonia,Macedonia
Brunei,Brunei Darussalam
Congo,Republic of Congo
East Timor,Timor-Leste
Gambia,The Gambia
Czechia,Czech Republic
""".strip().split('\n')
c_map_dict = {c.split(',')[0]:c.split(',')[1] for c in c_map}
cap_lan_long_list = []
for i,r in df_list[4].iterrows():
    ent = r['Entity']
    if ent in concap['CountryName'].values:
        con = ent
    elif ent in c_map_dict:
        con = c_map_dict[ent]
    else:
        continue
    row = concap.loc[concap['CountryName']==con,:].iloc[0]
    df_list[4].loc[i,'latitude'] = row['CapitalLatitude']
    df_list[4].loc[i,'longitude'] = row['CapitalLongitude']
print(df_list[4].head())

           Entity Code  Net imports - TWh  time   latitude   longitude
0           Italy  ITA              50.94  2024  41.900000   12.483333
1        Thailand  THA              37.08  2024  13.750000  100.516667
2  United Kingdom  GBR              33.19  2024  51.500000   -0.083333
3         Germany  DEU              25.45  2024  52.516667   13.400000
4          Brazil  BRA              15.11  2024 -15.783333  -47.916667


In [8]:
col_names = [['fuel','latitude','longitude','capacity_mw'],
             ['Type Name','Latitude','Longitude'],
             ['type','latitude','longitude','capacity_mw'],
             ['Product','Latitude','Longitude','Capacity_MWel'],
             ['Entity','latitude','longitude','Net imports - TWh'],
             ['Data Center','Latitude (generated)','Longitude (generated)'],
             ['ID','Latitude','Longitude','Rated Power (kW)'],
             ['TerminalName','Latitude','Longitude','Capacity']]
renames = ['','CCUS','SMR','Hydrogen','Interconnector',"Data Center","Storage","LNG"]

data_merge_list = []
for cn, df,rename in zip(col_names,df_list,renames):
    data = df.loc[:,cn].dropna(subset=cn[0])
    if rename != '':
        data.iloc[:,0] = rename
    data_merge_list.extend(data.values.tolist())
    # data_merge = pd.concat([data_merge,data.values],axis=0)
data_merge = pd.DataFrame(data_merge_list,columns=['type','latitude','longitude','capacity_mw'])
data_merge.fillna(0.1,inplace=True)
data_merge.to_csv('data_merge.csv',index=False)

In [9]:
data_merge

,type,latitude,longitude,capacity_mw
0,Hydro,32.322000,65.119000,33.00
1,Hydro,34.556000,69.478700,66.00
2,Hydro,34.641000,69.717000,100.00
3,Hydro,34.484700,70.363300,11.55
4,Hydro,35.941600,68.710000,6.00
...,...,...,...,...
12458,LNG,19.437793,105.768091,0.10
12459,LNG,20.823732,106.796686,0.70
12460,LNG,20.823732,106.796686,0.50
12461,LNG,-38.784331,-62.299546,5.84
